#### 1. Load images from `candidate_tiles` with Pillow

Folder name is the label. There is no label CSV.


##### 1.1 Find the data folder

Works from the project root or from `notebooks/`.


In [1]:
from collections import Counter
from pathlib import Path

from PIL import Image

DATASET_CANDIDATES = (
    Path("be-mlsys-assignment-dataset/candidate_tiles"),
    Path("../be-mlsys-assignment-dataset/candidate_tiles"),
)
data_dir = next((path for path in DATASET_CANDIDATES if path.is_dir()), None)
if data_dir is None:
    raise FileNotFoundError(
        "candidate_tiles not found. Run this notebook from the project root or from notebooks/."
    )

##### 1.2 Read each PNG and take the label from the folder

Open with Pillow. Store path, label, and size.


In [2]:
records = []
for class_dir in sorted(path for path in data_dir.iterdir() if path.is_dir()):
    label = class_dir.name
    for image_path in sorted(class_dir.glob("*.png")):
        with Image.open(image_path) as img:
            image = img.convert("RGB")
            width, height = image.size
        records.append(
            {
                "path": image_path,
                "label": label,
                "width": width,
                "height": height,
            }
        )

##### 1.3 Check what was loaded

7 classes, 150 images each.


In [3]:
counts = Counter(row["label"] for row in records)
print(f"loaded {len(records)} images from {data_dir.resolve()}")
for label, count in sorted(counts.items()):
    print(f"  {label}: {count}")


loaded 1050 images from C:\Users\DELL\Desktop\projects\Personal_projects\Satellite\be-mlsys-assignment-dataset\candidate_tiles
  AnnualCrop: 150
  Forest: 150
  Highway: 150
  Industrial: 150
  Residential: 150
  River: 150
  SeaLake: 150


#### 2. Split 80/20, stratified, with a fixed seed

Split `candidate_tiles` only. Each class stays 80/20.


##### 2.1 Fix the class order

Sorted folder names are class indexes 0 to 6.


In [4]:
classes = sorted({row["label"] for row in records})
label_to_index = {label: index for index, label in enumerate(classes)}
for row in records:
    row["class_index"] = label_to_index[row["label"]]

print(classes)


['AnnualCrop', 'Forest', 'Highway', 'Industrial', 'Residential', 'River', 'SeaLake']


##### 2.2 Split 80/20 with `train_test_split`

`stratify` keeps 120 train and 30 validation per class.


In [5]:
from sklearn.model_selection import train_test_split

train_records, val_records = train_test_split(
    records,
    test_size=0.2,
    random_state=18,
    stratify=[row["label"] for row in records],
)


##### 2.3 Check the split

120 and 30 per class. No path in both lists.


In [6]:
train_counts = Counter(row["label"] for row in train_records)
val_counts = Counter(row["label"] for row in val_records)
print(f"train {len(train_records)}  val {len(val_records)}")
for label in classes:
    print(f"  {label}: train {train_counts[label]}  val {val_counts[label]}")

overlap = {row["path"] for row in train_records} & {row["path"] for row in val_records}
print(f"shared paths: {len(overlap)}")


train 840  val 210
  AnnualCrop: train 120  val 30
  Forest: train 120  val 30
  Highway: train 120  val 30
  Industrial: train 120  val 30
  Residential: train 120  val 30
  River: train 120  val 30
  SeaLake: train 120  val 30
shared paths: 0


#### 3. Turn each PNG into a float tensor `3x64x64`

Divide pixels by 255. Convert train and validation separately.


##### 3.1 Convert one file

RGB, 64x64, float32, shape 3x64x64, values from 0 to 1.


In [7]:
import numpy as np
import torch
from PIL import Image

IMAGE_SIZE = 64

def image_to_tensor(path):
    with Image.open(path) as img:
        image = img.convert("RGB")
    if image.size != (IMAGE_SIZE, IMAGE_SIZE):
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE))
    pixels = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(pixels).permute(2, 0, 1)


##### 3.2 Stack train and validation separately

`train_x` from `train_records`. `val_x` from `val_records`.


In [8]:
def stack_split(split_records):
    images = [image_to_tensor(row["path"]) for row in split_records]
    labels = [row["class_index"] for row in split_records]
    return torch.stack(images), torch.tensor(labels, dtype=torch.long)

train_x, train_y = stack_split(train_records)
val_x, val_y = stack_split(val_records)


##### 3.3 Check shape, range, and separation

Train 840x3x64x64. Validation 210x3x64x64. Values between 0 and 1.


In [9]:
print("train", tuple(train_x.shape), train_x.dtype, float(train_x.min()), float(train_x.max()))
print("val", tuple(val_x.shape), val_x.dtype, float(val_x.min()), float(val_x.max()))

assert train_x.shape == (len(train_records), 3, IMAGE_SIZE, IMAGE_SIZE)
assert val_x.shape == (len(val_records), 3, IMAGE_SIZE, IMAGE_SIZE)
assert float(train_x.min()) >= 0.0 and float(train_x.max()) <= 1.0
assert float(val_x.min()) >= 0.0 and float(val_x.max()) <= 1.0
assert train_y.tolist() == [row["class_index"] for row in train_records]
assert val_y.tolist() == [row["class_index"] for row in val_records]
assert set(row["path"] for row in train_records).isdisjoint(row["path"] for row in val_records)
print("ok")


train (840, 3, 64, 64) torch.float32 0.0470588244497776 1.0
val (210, 3, 64, 64) torch.float32 0.07450980693101883 1.0
ok


#### 4. Define a small CNN

Three conv blocks, then a linear layer of size 7.


##### 4.1 One conv block

3x3 conv, ReLU, 2x2 pool. Channels 16, 32, 64. Size goes 64, 32, 16, 8.


In [10]:
import torch.nn as nn

def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


##### 4.2 Three blocks, then a linear layer of size 7

Flatten 64x8x8 into 7 class scores.


In [11]:
class TileCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 16),
            conv_block(16, 32),
            conv_block(32, 64),
        )
        self.classifier = nn.Linear(64 * 8 * 8, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model = TileCNN(num_classes=len(classes))


##### 4.3 Check the output shape

Four images should return shape 4x7.


In [13]:
with torch.no_grad():
    sample_scores = model(train_x[:4])

print(tuple(sample_scores.shape))
assert sample_scores.shape == (4, len(classes))
print("ok")


(4, 7)
ok


#### 5. Train on CPU

Cross-entropy, Adam, batch size 32, 15 epochs.


##### 5.1 Make batches

Shuffle training batches. Leave validation batches fixed.


In [14]:
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 32
EPOCHS = 15

train_loader = DataLoader(
    TensorDataset(train_x, train_y),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(val_x, val_y),
    batch_size=BATCH_SIZE,
    shuffle=False,
)
device = torch.device("cpu")
model.to(device)


TileCNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (classifier): Linear(in_features=4096, out_features=7, bias=True)
)

##### 5.2 Loss and optimizer

`CrossEntropyLoss` and Adam at learning rate 0.001.


In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


##### 5.3 Run 15 epochs

Print train loss and validation accuracy. Keep the best weights in `best_state`.


In [16]:
history = []
best_val_accuracy = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_seen = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        scores = model(images)
        loss = criterion(scores, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        total_seen += images.size(0)
    train_loss = total_loss / total_seen

    model.eval()
    correct = 0
    seen = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            pred = model(images).argmax(dim=1)
            correct += (pred == labels).sum().item()
            seen += labels.size(0)
    val_accuracy = correct / seen
    history.append(
        {"epoch": epoch, "train_loss": train_loss, "val_accuracy": val_accuracy}
    )
    print(f"epoch {epoch:02d}  train_loss {train_loss:.4f}  val_accuracy {val_accuracy:.3f}")
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


epoch 01  train_loss 1.7471  val_accuracy 0.333
epoch 02  train_loss 1.4209  val_accuracy 0.429
epoch 03  train_loss 1.3100  val_accuracy 0.519
epoch 04  train_loss 1.0822  val_accuracy 0.595
epoch 05  train_loss 0.9639  val_accuracy 0.505
epoch 06  train_loss 0.9509  val_accuracy 0.638
epoch 07  train_loss 0.8909  val_accuracy 0.543
epoch 08  train_loss 0.9030  val_accuracy 0.633
epoch 09  train_loss 0.7935  val_accuracy 0.624
epoch 10  train_loss 0.7353  val_accuracy 0.719
epoch 11  train_loss 0.6505  val_accuracy 0.714
epoch 12  train_loss 0.6106  val_accuracy 0.686
epoch 13  train_loss 0.5837  val_accuracy 0.719
epoch 14  train_loss 0.6285  val_accuracy 0.800
epoch 15  train_loss 0.4928  val_accuracy 0.762


##### 5.4 Check the run

Loss should drop. `best_state` is the epoch with the highest validation accuracy.


In [17]:
best = max(history, key=lambda row: row["val_accuracy"])
print(
    f"best epoch {best['epoch']}  val_accuracy {best['val_accuracy']:.3f}"
)
print(
    f"last epoch {history[-1]['epoch']}  val_accuracy {history[-1]['val_accuracy']:.3f}"
)
assert best_state is not None
assert len(history) == EPOCHS


best epoch 14  val_accuracy 0.800
last epoch 15  val_accuracy 0.762


#### 6. Keep the epoch with the best validation accuracy

Load `best_state` back onto the model.


##### 6.1 Load those weights


In [18]:
model.load_state_dict(best_state)
model.eval()
print(f"loaded best epoch, val_accuracy {best_val_accuracy:.3f}")


loaded best epoch, val_accuracy 0.800


##### 6.2 Accuracy and error on the holdout

Accuracy is correct labels out of 210. Error is 1 minus accuracy. Rows are the true class.


In [19]:
with torch.no_grad():
    val_scores = model(val_x)
    val_pred = val_scores.argmax(dim=1)

val_accuracy = (val_pred == val_y).float().mean().item()
val_error = 1.0 - val_accuracy
print(f"val_accuracy {val_accuracy:.3f}")
print(f"val_error    {val_error:.3f}")

confusion = torch.zeros(len(classes), len(classes), dtype=torch.long)
for true_index, guessed_index in zip(val_y.tolist(), val_pred.tolist()):
    confusion[true_index, guessed_index] += 1

print("true class     predicted counts in class order")
for index, name in enumerate(classes):
    counts = " ".join(f"{int(confusion[index, column]):4d}" for column in range(len(classes)))
    correct = int(confusion[index, index])
    total = int(confusion[index].sum())
    print(f"{name:12}  {counts}   {correct}/{total}")


val_accuracy 0.800
val_error    0.200
true class     predicted counts in class order
AnnualCrop      25    0    1    0    0    3    1   25/30
Forest           0   29    0    0    0    1    0   29/30
Highway          6    0    7    0   10    7    0   7/30
Industrial       0    0    0   27    3    0    0   27/30
Residential      0    0    1    2   27    0    0   27/30
River            1    0    2    0    1   26    0   26/30
SeaLake          0    1    0    0    0    2   27   27/30


#### 7. Pick a confidence cutoff from the holdout

Confidence is the highest of the 7 probabilities. Below the cutoff, status is uncertain.


##### 7.1 Confidence histogram

If no bin has 40% or more, cutoff stays 0.55. If the peak is at 0.7 or above, cutoff is 0.70.


In [20]:
with torch.no_grad():
    val_confidence = torch.softmax(val_scores, dim=1).max(dim=1).values

bin_counts = []
print("confidence  count")
for bin_index in range(10):
    low = bin_index / 10
    high = (bin_index + 1) / 10
    if bin_index == 9:
        count = int(((val_confidence >= low) & (val_confidence <= high)).sum())
    else:
        count = int(((val_confidence >= low) & (val_confidence < high)).sum())
    bin_counts.append(count)
    print(f"{low:.1f}-{high:.1f}     {count:4d}  {'#' * count}")

tallest = max(bin_counts)
peak_low = bin_counts.index(tallest) / 10
flat = tallest < 0.4 * len(val_confidence)
if flat:
    threshold = 0.55
    reason = "flat histogram, tallest bin is under 40%"
elif peak_low >= 0.7:
    threshold = 0.70
    reason = "tallest bin is at 0.7 or above"
else:
    threshold = 0.55
    reason = "the pile is below 0.7, so the cutoff stays 0.55"
print(f"threshold {threshold:.2f}  ({reason})")


confidence  count
0.0-0.1        0  
0.1-0.2        0  
0.2-0.3        0  
0.3-0.4        2  ##
0.4-0.5       16  ################
0.5-0.6       31  ###############################
0.6-0.7       36  ####################################
0.7-0.8       41  #########################################
0.8-0.9       28  ############################
0.9-1.0       56  ########################################################
threshold 0.55  (flat histogram, tallest bin is under 40%)


##### 7.2 What the cutoff does to the holdout

Count accepted and uncertain rows, and the accuracy of each group.


In [21]:
def group_accuracy(mask):
    if int(mask.sum()) == 0:
        return float("nan")
    return (val_pred[mask] == val_y[mask]).float().mean().item()

accepted = val_confidence >= threshold
uncertain = ~accepted
print(f"accepted  {int(accepted.sum()):3d}  accuracy {group_accuracy(accepted):.3f}")
print(f"uncertain {int(uncertain.sum()):3d}  accuracy {group_accuracy(uncertain):.3f}")
print(f"threshold {threshold:.2f}")


accepted  176  accuracy 0.824
uncertain  34  accuracy 0.676
threshold 0.55


#### 8. Save `artifacts/model.pt`

Save the weights and the class list.


##### 8.1 Write the checkpoint

This is the file the API loads.


In [22]:
import json

assert best_state is not None and "threshold" in dir()

artifacts_dir = Path("../artifacts")
if not artifacts_dir.is_dir():
    artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

model_path = artifacts_dir / "model.pt"
torch.save({"state_dict": best_state, "classes": classes}, model_path)
print(f"wrote {model_path.resolve()}")


wrote C:\Users\DELL\Desktop\projects\Personal_projects\Satellite\artifacts\model.pt


#### 9. Save `artifacts/meta.json`

Save the class list, image size, threshold, and preprocess name.


##### 9.1 Write the metadata file

`threshold` is the cutoff from step 7.


In [23]:
assert best_state is not None and "threshold" in dir()

meta = {
    "classes": classes,
    "image_size": IMAGE_SIZE,
    "threshold": threshold,
    "preprocess": "rgb_div_255",
}
meta_path = artifacts_dir / "meta.json"
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(f"wrote {meta_path.resolve()}")
print(meta)


wrote C:\Users\DELL\Desktop\projects\Personal_projects\Satellite\artifacts\meta.json
{'classes': ['AnnualCrop', 'Forest', 'Highway', 'Industrial', 'Residential', 'River', 'SeaLake'], 'image_size': 64, 'threshold': 0.55, 'preprocess': 'rgb_div_255'}


#### 10. Load from disk and score one tile

Load `model.pt` and run one image through it.


##### 10.1 Reload and predict one path


In [24]:
checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
loaded_classes = checkpoint["classes"]
reload_model = TileCNN(num_classes=len(loaded_classes))
reload_model.load_state_dict(checkpoint["state_dict"])
reload_model.eval()

sample_path = train_records[0]["path"]
with torch.no_grad():
    probs = torch.softmax(reload_model(image_to_tensor(sample_path).unsqueeze(0)), dim=1)[0]
pred_index = int(probs.argmax().item())
print(f"path {sample_path}")
print(f"predicted {loaded_classes[pred_index]}  confidence {float(probs[pred_index]):.3f}")


path ..\be-mlsys-assignment-dataset\candidate_tiles\SeaLake\SeaLake_37.png
predicted SeaLake  confidence 0.772
